# 財務常任委員会 議事録変換ツール 2026-03-1

---
### 使い方
1. **セル①** ライブラリ読み込み（最初に1回実行）
2. **セル②** 設定（入出力フォルダ・除去パターンなど）
3. **セル③** 変換コア関数の定義（1回実行）
4. **セル④** 実行（変換したいときに実行）
5. **セル⑤** 変換結果のプレビュー（任意）

In [ ]:
# セル① ライブラリ読み込み
import re
from pathlib import Path
from IPython.display import display, Markdown
print('✅ ライブラリ読み込み完了')


In [ ]:
# セル② 設定（ここだけ編集してください）

# 📂 入力フォルダ
# 例: INPUT_DIR = Path(r'C:\Users\yourname\Documents\gijiroku\zaimu')
INPUT_DIR = Path('.')

# 📂 出力フォルダ（None = INPUT_DIR と同じ）
OUTPUT_DIR = None

OUTPUT_PREFIX = 'Cleaned_'
INPUT_EXT     = '.txt'
OUTPUT_EXT    = '.md'
ENCODINGS     = ['cp932', 'utf-8-sig', 'utf-8', 'shift_jis']

# 🧹 ノイズ除去パターン
IGNORE_PATTERNS = [
    # 開会・成立
    # 見出し（空白始まりの短い区切り見出し）
    re.compile(r'^諸般の報告$'),
    re.compile(r'^付託議案の審査日程について$'),
    # 報告関連
    re.compile(r'^◯委員長[^　]*　この際、諸般の報告をいたします。$'),
    # 〔〕形式の行をまとめて除去（登壇・起立・異議なし等）
    re.compile(r'^〔.+〕$'),
    # 「御異議なしと認めます。」で文が終わる行のみ除去
    re.compile(r'御異議なしと認めます。$'),
    # 散会・閉会
    re.compile(r'本日はこれにて散会いたします'),
    re.compile(r'これより本日の会議を閉じます'),
    # 礼
    re.compile(r'（修　礼）'),
    # 時刻行（開会・開議・休憩・再開・散会・閉会）
    re.compile(r'^午[前後][０-９\d時分　]+[開休再散閉議会憩]'),
    # 「散　　　会」「閉　　　会」などの区切り行
    re.compile(r'^[散閉開休再][\s\u3000]+[会議憩]'),
]

print('✅ 設定完了')
print(f'   入力フォルダ: {INPUT_DIR.resolve()}')
print(f'   出力フォルダ: {(OUTPUT_DIR or INPUT_DIR).resolve()}')


In [ ]:
# セル③ 変換コア関数の定義

SPEAKER_RE = re.compile(
    r'^◯'
    r'('
    r'[^\s\u3000（]*（[^）]*）'
    r'|'
    r'[^\s\u3000]+'
    r')'
    r'(?:[\s\u3000]+(.*))?$'
)
SEPARATOR_RE  = re.compile(r'^[─━ー\-]{5,}')
BILL_RE       = re.compile(r'^\s*議案第[０-９\d]+号')
PAGE_RANGE_RE = re.compile(
    r'^[\d０-９]+[ページぺ]'
    r'|^第[０-９\d一二三四五六七八九十百]+款'
    r'|^第[０-９\d一二三四五六七八九十百]+項'
)
PAGE_RANGE_MAX_LEN = 50
DATETIME_RE  = re.compile(r'^午[前後][０-９\d]+時')
SIGNATURE_RE = re.compile(r'^\s*[令平][和成][０-９\d]+年[０-９\d]+月[０-９\d]+日\s*$')
REDACT_RE    = re.compile(r'＿＿+')


def read_file(path):
    for enc in ENCODINGS:
        try:
            return path.read_text(encoding=enc)
        except (UnicodeDecodeError, LookupError):
            continue
    return None


def is_noise(line):
    if SEPARATOR_RE.match(line): return True
    if '○' in line and len(line) > 20: return True
    if DATETIME_RE.match(line): return True
    if line.startswith('◯'): return False
    return any(p.search(line) for p in IGNORE_PATTERNS)


def format_speaker_line(line):
    m = SPEAKER_RE.match(line)
    if m:
        return f'**{m.group(1)}**: {(m.group(2) or "").strip()}'
    return line.replace('◯', '')


def apply_redaction(line):
    """＿が連続する箇所を（発言取消）に置き換える。"""
    return REDACT_RE.sub('（発言取消）', line)


def is_page_range_continuation(line):
    """PAGE_RANGE見出し行の継続行かどうか判定する。"""
    if not line: return False
    # 新しい見出しや発言行・行番号は継続行ではない
    if re.match(r'^(◯|─|━|〔|議案第)', line): return False
    if re.match(r'^\d+:\s*$', line): return False
    if PAGE_RANGE_RE.match(line): return False
    return True


def strip_signature(lines):
    """末尾の署名ブロック（令和〇年〇月〇日 から始まる）を除去する。"""
    for i, line in enumerate(lines):
        if SIGNATURE_RE.match(line):
            return lines[:i]
    return lines


def clean_lines(lines):
    lines = strip_signature(lines)
    cleaned = []
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        i += 1
        if not line: continue
        if re.match(r'^\d+:\s*$', line): continue
        if is_noise(line): continue
        line = re.sub(r'^\d+:\s*', '', line).strip()
        if BILL_RE.match(line):
            cleaned.append(f'## {line}')
            continue
        if PAGE_RANGE_RE.match(line) and len(line) <= PAGE_RANGE_MAX_LEN:
            # 折り返し継続行を連結する
            while i < len(lines):
                next_line = re.sub(r'^\d+:\s*', '', lines[i].strip())
                if is_page_range_continuation(next_line):
                    line = line + next_line
                    i += 1
                else:
                    break
            cleaned.append(f'### {line}')
            continue
        if line.startswith('◯'):
            line = format_speaker_line(line)
            line = apply_redaction(line)
        else:
            line = line.replace('◯', '')
        line = line.replace('＿＿＿', '').strip()
        if line:
            cleaned.append(line)
    return cleaned


def make_output_stem(src_stem):
    # パターン①: 末尾が「_YYYY-MM-DD」（アンダースコア区切り）
    parts = src_stem.rsplit('_', 1)
    if len(parts) == 2 and len(parts[1]) == 10 and parts[1][4] == '-' and parts[1][7] == '-':
        date = parts[1]
        body = parts[0].replace('_本文', '').replace('本文', '').rstrip('_')
        return date + '_' + body
    # パターン②: 末尾が「 YYYY-MM-DD」（半角スペース区切り）
    parts = src_stem.rsplit(' ', 1)
    if len(parts) == 2 and len(parts[1]) == 10 and parts[1][4] == '-' and parts[1][7] == '-':
        date = parts[1]
        body = parts[0].replace('\u3000本文', '').replace(' 本文', '').replace('本文', '').rstrip()
        return date + '_' + body
    return src_stem


def convert_file(src, dst):
    content = read_file(src)
    if content is None:
        print(f'  ❌ 読み込み失敗: {src.name}')
        return False
    out_stem = make_output_stem(src.stem)
    lines    = clean_lines(content.splitlines())
    md_text  = '# ' + out_stem + '\n\n' + '\n\n'.join(lines)
    dst.parent.mkdir(parents=True, exist_ok=True)
    dst.write_text(md_text, encoding='utf-8-sig')
    print(f'  ✅ {src.name}')
    print(f'      → {dst.name}')
    return True


print('✅ 関数定義完了')


In [ ]:
# セル④ 実行
out_dir = OUTPUT_DIR or INPUT_DIR
targets = sorted(
    p for p in INPUT_DIR.glob(f'*{INPUT_EXT}')
    if not p.name.startswith(OUTPUT_PREFIX)
)
if not targets:
    print(f'⚠️  対象の {INPUT_EXT} ファイルが見つかりませんでした。')
    print(f'   INPUT_DIR を確認してください: {INPUT_DIR.resolve()}')
else:
    print(f'📋 変換対象: {len(targets)} 件\n')
    success, failure = 0, 0
    for src in targets:
        dst = out_dir / (OUTPUT_PREFIX + make_output_stem(src.stem) + OUTPUT_EXT)
        if convert_file(src, dst):
            success += 1
        else:
            failure += 1
    print(f'\n{"="*40}')
    print(f'✅ 成功: {success} 件  ❌ 失敗: {failure} 件')
    print(f'出力先: {out_dir.resolve()}')


In [ ]:
# セル⑤ 変換結果のプレビュー（任意）
PREVIEW_TARGET = ''   # ← 入力ファイル名（拡張子なし）を入力
PREVIEW_LINES  = 80

if not PREVIEW_TARGET:
    print('ℹ️  PREVIEW_TARGET にファイル名（拡張子なし）を入力してください。')
else:
    out_dir = OUTPUT_DIR or INPUT_DIR
    md_path = out_dir / (OUTPUT_PREFIX + make_output_stem(PREVIEW_TARGET) + OUTPUT_EXT)
    if not md_path.exists():
        print(f'❌ ファイルが見つかりません: {md_path}')
    else:
        text    = md_path.read_text(encoding='utf-8-sig')
        preview = '\n'.join(text.splitlines()[:PREVIEW_LINES])
        display(Markdown(preview))
